In [28]:
!pip install lightgbm

In [36]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.compose import ColumnTransformer
import lightgbm as lgb
from lightgbm import LGBMRegressor

---
## <u>Load Dataset</u>

In [37]:
data = sns.load_dataset("taxis")
data["pickup"].dtype # date time 
data["dropoff"].dtype # date time 
data["payment"].nunique() # 2 categories
data["pickup_zone"].nunique() # 194
data["dropoff_zone"].nunique() # 203
data["pickup_borough"].nunique() # 4
data["dropoff_borough"].nunique() # 5
data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6433 entries, 0 to 6432
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   pickup           6433 non-null   datetime64[ns]
 1   dropoff          6433 non-null   datetime64[ns]
 2   passengers       6433 non-null   int64         
 3   distance         6433 non-null   float64       
 4   fare             6433 non-null   float64       
 5   tip              6433 non-null   float64       
 6   tolls            6433 non-null   float64       
 7   total            6433 non-null   float64       
 8   color            6433 non-null   object        
 9   payment          6389 non-null   object        
 10  pickup_zone      6407 non-null   object        
 11  dropoff_zone     6388 non-null   object        
 12  pickup_borough   6407 non-null   object        
 13  dropoff_borough  6388 non-null   object        
dtypes: datetime64[ns](2), float64(5), int64(

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


---
## <u>Train Test Split</u>

In [38]:
# We drop 'total', 'fare', 'tip', and 'tolls' because they leak the final price i.e. total = fare + tip + tolls
X = data.drop(columns=['total', 'fare', 'tip', 'tolls']) 
y = data['total']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
## <u>Feature Encoding</u>

In [39]:
# 1. HANDLE DATETIME COLUMNS

# ML models cannot read date strings like "2019-03-23 00:03:22" so We convert them to 'datetime' objects to unlock the '.dt' paramter which allows us to slice the
# timestamps into meaningful numerical components.

for df in [X_train, X_test]:
    # Convert string text to make pandas Timestamp objects
    df["pickup"] = pd.to_datetime(df["pickup"])
    df["dropoff"] = pd.to_datetime(df["dropoff"])
    
    # Extract numerical features
    df["pickup_hour"] = df["pickup"].dt.hour            # Returns 0 to 23# Returns 0 to 23 (rush hour vs midnight matters)
    df["droppoff_hour"] = df["dropoff"].dt.hour         # Returns 0 to 23# Returns 0 to 23 (rush hour vs midnight matters)
    df["pickup_dayofweek"] = df["pickup"].dt.dayofweek  # Returns 0 to 6 (weekdays vs weekends)
    
    # Subtracting two timestamps gives a 'Timedelta' object.
    # We extract total seconds and divide by 60 to convert it into raw minutes.
    df["trip_duration_mins"] = (df["dropoff"] - df["pickup"]).dt.total_seconds() / 60.0

# 2. DROP THE RAW DATETIME COLUMNS 

X_train = X_train.drop(columns=["pickup", "dropoff"])
X_test = X_test.drop(columns=["pickup", "dropoff"])

# 3. TARGET ENCODING (For High-Cardinality Categories) and ONE-HOT ENCODING (For Low-Cardinality Categories)

# "pickup_zone" has 194 categories. One-Hot encoding would make 194 new columns, Instead, Target Encoding replaces the category name with a single number i.e. the
# average target value (average taxi fare) for that specific category. Scikit-Learn's TargetEncoder handles "noisy" or rare categories automatically by shrinking 
# them toward the global average so they don't overfit.

# Example: If trips starting in "JFK Airport" average a total fare of $55, "JFK Airport" is replaced by 55.0. 

high_card_cols = ['pickup_zone', 'dropoff_zone']                          # columns with toooooo many categories, we use target encoder here
low_card_cols = ['payment', 'pickup_borough', 'dropoff_borough', 'color'] # columns with managable categories, we use one hot encoder her
numeric_cols = ['passengers', 'distance']

preprocessor = ColumnTransformer(
    transformers = [
        ("tar", TargetEncoder(smooth="auto", random_state=42), high_card_cols),
        ("scaler" , StandardScaler(), numeric_cols),
        ("ohe", OneHotEncoder(sparse_output=False, handle_unknown='ignore'), low_card_cols)
    ],
    remainder='passthrough' # This prevents our other features excluding high_card_cols/low_card_cols from being dropped, example : df['trip_duration_mins']
)

# 4. SET THE OUTPUT OF PROCESSOR AS PANDAS DATAFRAME

preprocessor.set_output(transform="pandas")

# 5. USE THE PREPROCESSOR ON THE COLUMNS WITH HIGH AND LOW CARDINALITY

X_train = preprocessor.fit_transform(X_train, y_train)  # for target encoder to work we pass the y_train values as well
X_test = preprocessor.transform(X_test)

---
## <u>Create, Train and Predict</u>

In [40]:
lgbmr_model = LGBMRegressor(random_state = 42, verbose = -1)

lgbmr_model.fit(X_train, y_train)

y_train_pred = lgbmr_model.predict(X_train)
y_test_pred = lgbmr_model.predict(X_test)

---
## <u>Evaluate</u>

In [41]:
print("For LightGBM Regressor (baseline) :-\n")

print("Training scores :-")
print("Train R2_score : ", r2_score(y_train, y_train_pred))


print("\nTesting scores :-")
print("Test R2_score : ", r2_score(y_test, y_test_pred))

For LightGBM Regressor (baseline) :-

Training scores :-
Train R2_score :  0.9507081391163289

Testing scores :-
Test R2_score :  0.8766073637962857


---
## <u>Hyperparameter Tuning</u>

In [43]:
# 1. Initialise the steps and make the pipeline

steps = [("lgbmr", LGBMRegressor(random_state = 42, verbose = -1, n_jobs = 1))]
pipeline = Pipeline(steps)

# 2. define the paramter grid 

param_grid = {
    "lgbmr__max_depth" : [2, 4, 6],
    "lgbmr__learning_rate" : [0.01 ,0.05, 0.1], 
    "lgbmr__n_estimators" : [200, 250, 300],
    "lgbmr__subsample" : [0.5, 0.6, 0.7],
    "lgbmr__reg_alpha" : [5, 10, 15], 
    "lgbmr__reg_lambda" : [5, 10, 15], 
    "lgbmr__min_child_samples" : [50, 75, 100], 
    "lgbmr__num_leaves" : [14, 16, 18] # to cure overfitting, this is the rule : num_leaves must always be smaller than 2^max_depth
}

# 3. cross validation blueprint

lightGBM_regressor_cv = RandomizedSearchCV(
    pipeline,
    param_grid,
    cv = 3,
    n_iter = 50,
    n_jobs = 1,
    verbose=0 
)

# 4. make and train the model

lightGBM_regressor_cv.fit(X_train, y_train)

# 5. make predictions
y_train_pred = lightGBM_regressor_cv.predict(X_train)
y_test_pred = lightGBM_regressor_cv.predict(X_test)

# 6. evaluate 

print("For LightGBM Regressor (hyperparameter tuning) :-\n")

print("Best parameters found: ", lightGBM_regressor_cv.best_params_)

print("\nTraining scores :-")
print("Train R2_score : ", r2_score(y_train, y_train_pred))


print("\nTesting scores :-")
print("Test R2_score : ", r2_score(y_test, y_test_pred))

For LightGBM Regressor (hyperparameter tuning) :-

Best parameters found:  {'lgbmr__subsample': 0.7, 'lgbmr__reg_lambda': 15, 'lgbmr__reg_alpha': 5, 'lgbmr__num_leaves': 14, 'lgbmr__n_estimators': 300, 'lgbmr__min_child_samples': 50}

Training scores :-
Train R2_score :  0.9386331931143093

Testing scores :-
Test R2_score :  0.8562736191836379
